# ML Training Pipeline for Partial Mueller Features (4x3)

Use all rows and first 3 columns (12 features).

This notebook follows the same training pipeline as `notebooks/training/training.ipynb` (Brain + Cervix + AFMMM, class checks, split, MLP, XGBoost, CatBoost).


In [1]:
import os
import time
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import xgboost as xgb
from catboost import CatBoostClassifier, Pool

sys.path.append('..')
from src.utils.file_paths import file_paths
from src.utils.visualisation import plot_and_save_confusion_matrix

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

VARIANT = '4x3'
VARIANT_TAG = VARIANT.replace('_', '-')

resultspath = file_paths.model_save_path.parent / 'results' / 'partial_experiments' / VARIANT
resultspath.mkdir(parents=True, exist_ok=True)

labels = ['Not PR', 'PR']
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('Device:', device)
print('Results path:', resultspath)

Device: mps
Results path: /Users/chaechae/Desktop/EP_Code/partialPr/results/partial_experiments/4x3


## 1. Load the Prepared Data


In [2]:
X_brain_full = np.load(file_paths.brain_processed_path / 'merged_all_X.npy')
y_brain      = np.load(file_paths.brain_processed_path / 'merged_all_y.npy')

X_cervix_full = np.load(file_paths.cervix_processed_path / 'merged_all_X.npy')
y_cervix      = np.load(file_paths.cervix_processed_path / 'merged_all_y.npy')

X_afmmm_full = np.load(file_paths.afmmm_processed_path / 'merged_all_X.npy')
y_afmmm      = np.load(file_paths.afmmm_processed_path / 'merged_all_y.npy')

print('brain_full: ', X_brain_full.shape, y_brain.shape)
print('cervix_full:', X_cervix_full.shape, y_cervix.shape)
print('afmmm_full: ', X_afmmm_full.shape, y_afmmm.shape)


brain_full:  (31, 388, 516, 16) (31, 388, 516)
cervix_full: (19, 600, 800, 16) (19, 600, 800)
afmmm_full:  (47, 500, 500, 16) (47, 500, 500)


## 2. Prepare Data for Training


In [3]:
def to_m44(X):
    X = np.asarray(X)
    if X.ndim == 4 and X.shape[-2:] == (4, 4):
        return X
    if X.ndim == 4 and X.shape[-1] == 16:
        return X.reshape(*X.shape[:3], 4, 4)
    if X.ndim == 4 and X.shape[-1] == 12:
        # legacy representation: 4x3 flattened per pixel
        m = np.zeros((*X.shape[:3], 4, 4), dtype=X.dtype)
        m[:, :, :, :3] = X.reshape(*X.shape[:3], 4, 3)
        return m
    if X.ndim == 3 and X.shape[-1] == 16:
        return X.reshape(*X.shape[:2], 4, 4)
    if X.ndim == 3 and X.shape[-1] == 12:
        m = np.zeros((*X.shape[:2], 4, 4), dtype=X.dtype)
        m[:, :, :, :3] = X.reshape(*X.shape[:2], 4, 3)
        return m
    raise ValueError(f'Unsupported X shape for conversion to 4x4: {X.shape}')


def extract_partial_features(X_full, variant):
    M = to_m44(X_full)

    if variant == '4x3':
        return M[:, :, :, :, :3].reshape(*M.shape[:3], 12)

    if variant == '3x3':
        return M[:, :, :, :3, :3].reshape(*M.shape[:3], 9)

    raise ValueError(f'Unknown variant: {variant}')


def flatten(X, y):
    ns, H, W, F = X.shape
    return X.reshape(-1, F), y.reshape(-1)

X_brain = extract_partial_features(X_brain_full, VARIANT)
X_cervix = extract_partial_features(X_cervix_full, VARIANT)
X_afmmm = extract_partial_features(X_afmmm_full, VARIANT)

print('brain: ', X_brain.shape)
print('cervix:', X_cervix.shape)
print('afmmm: ', X_afmmm.shape)

Xb_flat, yb_flat = flatten(X_brain, y_brain)
Xc_flat, yc_flat = flatten(X_cervix, y_cervix)
Xa_flat, ya_flat = flatten(X_afmmm, y_afmmm)

X_flat = np.vstack([Xb_flat, Xc_flat, Xa_flat])
y_flat = np.concatenate([yb_flat, yc_flat, ya_flat])

print('After merge:', X_flat.shape, y_flat.shape)

brain:  (31, 388, 516, 12)
cervix: (19, 600, 800, 12)
afmmm:  (47, 500, 500, 12)
After merge: (27076448, 12) (27076448,)


## 3. Explore Data Balance


In [5]:
def get_class_distribution(y):
    y_flat = y.reshape(-1)
    values, counts = np.unique(y_flat, return_counts=True)
    dist = dict(zip(values, counts))
    total = sum(counts)
    pct_0 = dist.get(0, 0) / total * 100
    pct_1 = dist.get(1, 0) / total * 100
    return pct_0, pct_1, total

brain_0, brain_1, brain_total = get_class_distribution(y_brain)
cervix_0, cervix_1, cervix_total = get_class_distribution(y_cervix)
afmmm_0, afmmm_1, afmmm_total = get_class_distribution(y_afmmm)

print(f'Brain  -> PR: {brain_1:.2f}%, Not PR: {brain_0:.2f}%')
print(f'Cervix -> PR: {cervix_1:.2f}%, Not PR: {cervix_0:.2f}%')
print(f'AFMMM  -> PR: {afmmm_1:.2f}%, Not PR: {afmmm_0:.2f}%')

unique_values, counts = np.unique(y_flat, return_counts=True)
class_distribution = dict(zip(unique_values, counts))
print('Overall class distribution:')
for value, count in class_distribution.items():
    print(f'Class {value}: {count} ({count / len(y_flat) * 100:.2f}%)')


Brain  -> PR: 98.05%, Not PR: 1.95%
Cervix -> PR: 18.39%, Not PR: 81.61%
AFMMM  -> PR: 29.91%, Not PR: 70.09%
Overall class distribution:
Class 0.0: 15799210 (58.35%)
Class 1.0: 11277238 (41.65%)


## 4. Split Data for Training, Validation, and Testing


In [6]:
mask_0 = (y_flat == 0)
mask_1 = (y_flat == 1)
indices_0 = np.where(mask_0)[0]
indices_1 = np.where(mask_1)[0]

sample_size_0 = max(1, len(indices_0) // 10)
sample_size_1 = max(1, len(indices_1) // 10)

sampled_indices_0 = np.random.choice(indices_0, size=sample_size_0, replace=False)
sampled_indices_1 = np.random.choice(indices_1, size=sample_size_1, replace=False)

sampled_indices = np.concatenate([sampled_indices_0, sampled_indices_1])
np.random.shuffle(sampled_indices)

X_flat_sampled = X_flat[sampled_indices]
y_flat_sampled = y_flat[sampled_indices]

print(f'Original data size: {len(X_flat):,}')
print(f'Sampled data size: {len(X_flat_sampled):,}')

X_train, X_test, y_train, y_test = train_test_split(
    X_flat_sampled, y_flat_sampled, test_size=0.2, random_state=RANDOM_SEED, stratify=y_flat_sampled
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=RANDOM_SEED, stratify=y_train
)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val:  ', X_val.shape, 'y_val:  ', y_val.shape)
print('X_test: ', X_test.shape, 'y_test: ', y_test.shape)


Original data size: 27,076,448
Sampled data size: 2,707,644
X_train: (1624586, 12) y_train: (1624586,)
X_val:   (541529, 12) y_val:   (541529,)
X_test:  (541529, 12) y_test:  (541529,)


## 5. Per-pixel Classification with an MLP


In [7]:
train_dataset_cls = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32),
)
val_dataset_cls = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32),
)
test_dataset_cls = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32),
)

batch_size_cls   = 8192
train_loader_cls = DataLoader(train_dataset_cls, batch_size=batch_size_cls, shuffle=True)
val_loader_cls   = DataLoader(val_dataset_cls, batch_size=batch_size_cls)
test_loader_cls  = DataLoader(test_dataset_cls, batch_size=batch_size_cls)


class PixelMLP(nn.Module):
    def __init__(self, in_features, hidden_sizes=(128, 64, 32)):
        super().__init__()
        layers = []
        prev = in_features
        for h in hidden_sizes:
            layers += [
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(0.30),
            ]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


clf_model = PixelMLP(in_features=X_train.shape[1]).to(device)

pos_fraction_cls = y_train.mean()
neg_fraction_cls = 1.0 - pos_fraction_cls
pos_weight_val   = neg_fraction_cls / max(pos_fraction_cls, 1e-8)
criterion_cls    = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], dtype=torch.float32, device=device))

optimizer_cls = optim.Adam(clf_model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler_cls = optim.lr_scheduler.ReduceLROnPlateau(optimizer_cls, mode='min', factor=0.1, patience=3)


def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.set_grad_enabled(train):
        for data, target in loader:
            data, target = data.to(device), target.to(device)

            if train:
                optimizer_cls.zero_grad()

            logits = model(data)
            loss = criterion_cls(logits, target)

            if train:
                loss.backward()
                optimizer_cls.step()

            total_loss += loss.item() * data.size(0)

            probs = torch.sigmoid(logits)
            y_true.append(target.cpu().numpy())
            y_pred.append((probs > 0.5).float().cpu().numpy())

    y_true = np.concatenate(y_true).ravel()
    y_pred = np.concatenate(y_pred).ravel()

    avg_loss = total_loss / len(loader.dataset)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    return avg_loss, acc, prec, rec, f1


best_val = float('inf')
epochs_cls = 25
for epoch in range(1, epochs_cls + 1):
    tr_loss, tr_acc, _, _, _ = run_epoch(clf_model, train_loader_cls, train=True)
    vl_loss, vl_acc, vl_prec, vl_rec, vl_f1 = run_epoch(clf_model, val_loader_cls, train=False)
    scheduler_cls.step(vl_loss)

    print(f'[MLP Epoch {epoch:02d}/{epochs_cls}] train loss {tr_loss:.4f} | val loss {vl_loss:.4f} | val F1 {vl_f1:.4f}')

    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(clf_model.state_dict(), file_paths.model_save_path / f'best_pixel_mlp_{VARIANT}.pth')


y_true_list, y_pred_list, y_probs_list = [], [], []
clf_model.eval()
with torch.no_grad():
    for data, target in test_loader_cls:
        data = data.to(device)
        logits = clf_model(data)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        preds = (probs > 0.5).astype(int)
        y_pred_list.append(preds)
        y_probs_list.append(probs)
        y_true_list.append(target.cpu().numpy().ravel())

y_true_all = np.concatenate(y_true_list)
y_pred_all = np.concatenate(y_pred_list)
y_probs_all = np.concatenate(y_probs_list)

mlp_results = {
    'model': f'PixelMLP_{VARIANT_TAG}',
    'n_features': int(X_train.shape[1]),
    'accuracy': accuracy_score(y_true_all, y_pred_all),
    'precision': precision_score(y_true_all, y_pred_all, zero_division=0),
    'recall': recall_score(y_true_all, y_pred_all, zero_division=0),
    'f1': f1_score(y_true_all, y_pred_all, zero_division=0),
    'roc_auc': roc_auc_score(y_true_all, y_probs_all),
}

pd.DataFrame([mlp_results]).to_csv(resultspath / f'pixel_mlp_metrics_{VARIANT}.csv', index=False)
print('Saved MLP metrics ->', resultspath / f'pixel_mlp_metrics_{VARIANT}.csv')

plot_and_save_confusion_matrix(
    y_true=y_test.ravel(),
    y_pred=y_pred_all,
    labels=labels,
    title=f'Pixel-MLP Confusion Matrix ({VARIANT_TAG})',
    save_path=resultspath / f'mlp_confusion_matrix_{VARIANT}.png'
)


[MLP Epoch 01/25] train loss 0.2307 | val loss 0.1024 | val F1 0.9688
[MLP Epoch 02/25] train loss 0.1087 | val loss 0.0810 | val F1 0.9733
[MLP Epoch 03/25] train loss 0.0943 | val loss 0.0775 | val F1 0.9731
[MLP Epoch 04/25] train loss 0.0889 | val loss 0.0728 | val F1 0.9752
[MLP Epoch 05/25] train loss 0.0861 | val loss 0.0743 | val F1 0.9728
[MLP Epoch 06/25] train loss 0.0826 | val loss 0.0709 | val F1 0.9741
[MLP Epoch 07/25] train loss 0.0812 | val loss 0.0679 | val F1 0.9757
[MLP Epoch 08/25] train loss 0.0789 | val loss 0.0666 | val F1 0.9758
[MLP Epoch 09/25] train loss 0.0783 | val loss 0.0666 | val F1 0.9753
[MLP Epoch 10/25] train loss 0.0767 | val loss 0.0677 | val F1 0.9754
[MLP Epoch 11/25] train loss 0.0758 | val loss 0.0645 | val F1 0.9762
[MLP Epoch 12/25] train loss 0.0751 | val loss 0.0666 | val F1 0.9760
[MLP Epoch 13/25] train loss 0.0745 | val loss 0.0635 | val F1 0.9763
[MLP Epoch 14/25] train loss 0.0747 | val loss 0.0631 | val F1 0.9765
[MLP Epoch 15/25] tr

## 6. XGBoost Gradient-Boosted Trees


In [8]:
dtrain = xgb.DMatrix(X_train, label=y_train)  # PR class weight is applied once via scale_pos_weight
dval   = xgb.DMatrix(X_val, label=y_val)
dtest  = xgb.DMatrix(X_test, label=y_test)

params = dict(
    objective='binary:logistic',
    eval_metric='logloss',
    eta=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight_val,
    nthread=-1,
    seed=RANDOM_SEED,
)

print('Training XGBoost...')
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100,
)

y_proba_xgb = bst.predict(dtest)
y_hat_xgb = (y_proba_xgb > 0.5).astype(int)

xgb_results = {
    'model': f'XGBoost_{VARIANT_TAG}',
    'n_features': int(X_train.shape[1]),
    'accuracy': accuracy_score(y_test, y_hat_xgb),
    'precision': precision_score(y_test, y_hat_xgb, zero_division=0),
    'recall': recall_score(y_test, y_hat_xgb, zero_division=0),
    'f1': f1_score(y_test, y_hat_xgb, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba_xgb),
}

bst.save_model(str(file_paths.model_save_path / f'pixel_xgb_{VARIANT}.json'))
pd.DataFrame([xgb_results]).to_csv(resultspath / f'xgb_metrics_{VARIANT}.csv', index=False)
print('Saved XGBoost metrics ->', resultspath / f'xgb_metrics_{VARIANT}.csv')

plot_and_save_confusion_matrix(
    y_true=y_test.ravel(),
    y_pred=y_hat_xgb,
    labels=labels,
    title=f'XGBoost Confusion Matrix ({VARIANT_TAG})',
    save_path=resultspath / f'xgb_confusion_matrix_{VARIANT}.png'
)


Training XGBoost...
[0]	val-logloss:0.63600
[100]	val-logloss:0.06947
[200]	val-logloss:0.05690
[300]	val-logloss:0.05264
[400]	val-logloss:0.05048
[500]	val-logloss:0.04924
[600]	val-logloss:0.04837
[700]	val-logloss:0.04772
[800]	val-logloss:0.04723
[900]	val-logloss:0.04692
[1000]	val-logloss:0.04659
[1100]	val-logloss:0.04636
[1200]	val-logloss:0.04622
[1300]	val-logloss:0.04608
[1400]	val-logloss:0.04600
[1500]	val-logloss:0.04592
[1600]	val-logloss:0.04585
[1700]	val-logloss:0.04582
[1800]	val-logloss:0.04575
[1900]	val-logloss:0.04572
[1995]	val-logloss:0.04571
Saved XGBoost metrics -> /Users/chaechae/Desktop/EP_Code/partialPr/results/partial_experiments/4x3/xgb_metrics_4x3.csv


## 7. CatBoost Classifier


In [9]:
train_pool = Pool(X_train, y_train)  # PR class weight is applied once via class_weights
val_pool = Pool(X_val, y_val)

cat = CatBoostClassifier(
    loss_function='Logloss',
    depth=8,
    learning_rate=0.05,
    iterations=2000,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_SEED,
    eval_metric='Logloss',
    verbose=200,
    early_stopping_rounds=100,
    class_weights=[1.0, pos_weight_val],  # class 0=non-PR, class 1=PR
)

print('Training CatBoost...')
cat.fit(train_pool, eval_set=val_pool)

y_proba_cb = cat.predict_proba(X_test)[:, 1]
y_pred_cb = (y_proba_cb > 0.5).astype(int)

cb_results = {
    'model': f'CatBoost_{VARIANT_TAG}',
    'n_features': int(X_train.shape[1]),
    'accuracy': accuracy_score(y_test, y_pred_cb),
    'precision': precision_score(y_test, y_pred_cb, zero_division=0),
    'recall': recall_score(y_test, y_pred_cb, zero_division=0),
    'f1': f1_score(y_test, y_pred_cb, zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_proba_cb),
}

cat.save_model(file_paths.model_save_path / f'pixel_catboost_{VARIANT}.cbm')
pd.DataFrame([cb_results]).to_csv(resultspath / f'catboost_metrics_{VARIANT}.csv', index=False)
print('Saved CatBoost metrics ->', resultspath / f'catboost_metrics_{VARIANT}.csv')

plot_and_save_confusion_matrix(
    y_true=y_test.ravel(),
    y_pred=y_pred_cb,
    labels=labels,
    title=f'CatBoost Confusion Matrix ({VARIANT_TAG})',
    save_path=resultspath / f'catboost_confusion_matrix_{VARIANT}.png'
)


Training CatBoost...
0:	learn: 0.5579624	test: 0.5594783	best: 0.5594783 (0)	total: 120ms	remaining: 4m
200:	learn: 0.0499734	test: 0.0569786	best: 0.0569786 (200)	total: 11.1s	remaining: 1m 39s
400:	learn: 0.0444329	test: 0.0511294	best: 0.0511294 (400)	total: 19.3s	remaining: 1m 16s
600:	learn: 0.0415270	test: 0.0484556	best: 0.0484556 (600)	total: 27.5s	remaining: 1m 3s
800:	learn: 0.0397144	test: 0.0470057	best: 0.0470057 (800)	total: 35.6s	remaining: 53.3s
1000:	learn: 0.0382827	test: 0.0459678	best: 0.0459678 (1000)	total: 43.8s	remaining: 43.7s
1200:	learn: 0.0371076	test: 0.0452306	best: 0.0452306 (1200)	total: 52.1s	remaining: 34.6s
1400:	learn: 0.0361460	test: 0.0446982	best: 0.0446982 (1400)	total: 1m	remaining: 25.8s
1600:	learn: 0.0352751	test: 0.0442797	best: 0.0442797 (1600)	total: 1m 8s	remaining: 17s
1800:	learn: 0.0345420	test: 0.0439511	best: 0.0439511 (1800)	total: 1m 16s	remaining: 8.46s
1999:	learn: 0.0341213	test: 0.0437839	best: 0.0437839 (1999)	total: 1m 24s	re

## 8. Summary


In [10]:
summary = pd.DataFrame([mlp_results, xgb_results, cb_results])
summary.to_csv(resultspath / f'summary_metrics_{VARIANT}.csv', index=False)
summary


,model,n_features,accuracy,precision,recall,f1,roc_auc
0,PixelMLP_4x3,12,0.979100,0.973390,0.976515,0.974950,0.998231
1,XGBoost_4x3,12,0.981698,0.985352,0.970485,0.977862,0.998504
2,CatBoost_4x3,12,0.980618,0.991912,0.961303,0.976367,0.998440
